<a href="https://colab.research.google.com/github/Dineshseervi/AI_IIITM/blob/main/week_12/00_foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W12 — Foundations: How Vector DBs and MCP Actually Work

**Level:** Beginner (run this *before* `01_principles` and `02_patterns`)
**Stack:** sentence-transformers + numpy + the `mcp` SDK — no API key needed
**Domain:** Personal Knowledge Base (note-taking assistant)
**Time:** ~45 min

## Why this notebook exists

`01_principles` opens a Chroma vector store in three lines, and `02_patterns` spins up a FastMCP server with a decorator. Both *work*, but they hide the machinery. If the first time you meet "embeddings" is `Chroma(embedding_function=...)`, the abstraction is doing your thinking for you.

This notebook slows down and builds the two ideas from the ground up, using the **real** libraries but one small step at a time:

- **Part A — Vector databases.** What an embedding actually is (you will print one), how "similarity" is just a number you can compute by hand, and a 12-line semantic search with *no database at all*. Then: exactly what Chroma adds on top.
- **Part B — MCP.** What "calling a tool" really means, the two verbs every tool protocol needs (`list_tools` / `call_tool`), what a JSON-RPC message looks like, and a minimal real FastMCP server. Then: exactly what MCP adds on top.

By the end, the three-line Chroma setup and the FastMCP decorator in the next notebooks will read as *conveniences*, not magic.

## Learning objectives

- Explain an embedding as a fixed-length vector and inspect its raw numbers
- Compute cosine similarity by hand and use it to rank text by meaning
- Write a working semantic search in ~12 lines without any vector DB
- State precisely what Chroma adds over a hand-rolled search
- Describe tool-calling as a `list_tools` / `call_tool` handshake
- Read a JSON-RPC request/response and map it to that handshake
- Run a minimal FastMCP server and call it through the real SDK
- State precisely what MCP adds over a hand-rolled tool registry

## Setup — lighter than the rest of the week

Most of this notebook is pure Python + numpy + a small local embedding model. **No `TOGETHER_API_KEY` or `OPENAI_API_KEY` is required** — handy if you want to learn the ideas before wiring up providers.

You do need two packages (already in `_global/requirements.txt`):

```bash
pip install sentence-transformers "mcp[cli]"
```

The first run of the embedding model downloads ~80 MB (`all-MiniLM-L6-v2`), then it is cached.

In [ ]:
# Part A imports — embeddings + numpy
!pip install sentence-transformers "mcp[cli]"

import numpy as np

# We call sentence-transformers DIRECTLY here (not via LangChain's HuggingFaceEmbeddings
# wrapper, which 01_principles uses). Calling it raw lets us see the actual numbers.
from sentence_transformers import SentenceTransformer

print("numpy:", np.__version__)
print("Loading embedding model (first run downloads ~80 MB)...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Model ready. Output dimension:", model.get_sentence_embedding_dimension())

numpy: 2.0.2
Loading embedding model (first run downloads ~80 MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model ready. Output dimension: 384


/tmp/ipykernel_3174/1331393935.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model ready. Output dimension:", model.get_sentence_embedding_dimension())


# Part A — Vector databases from the ground up

## A1 — What is an embedding?

An **embedding** is a function that turns a piece of text into a fixed-length list of numbers (a *vector*). The model was trained so that texts with similar *meaning* land at nearby points in this numeric space — even if they share no words.

`all-MiniLM-L6-v2` maps any sentence to a vector of **384 floats**. "I love pizza" and "Pizza is my favourite food" share almost no vocabulary, but their 384-number vectors point in nearly the same direction. That is the whole trick: meaning becomes geometry.

Let's stop describing it and print one.

In [ ]:
sentences = [
    "I love eating pizza on Fridays",
    "Pizza is my favourite food",
    "The stock market dropped sharply today",
    "current year is 2026 will not eat pizza much at home",
]

vectors = model.encode(sentences)   # -> numpy array, shape (3, 384)

print("Type:", type(vectors).__name__)
print("Shape:", vectors.shape, "  (3 sentences, 384 numbers each)")
print()
print("First sentence, first 8 of its 384 numbers:")
print(np.round(vectors[0][:8], 4))

Type: ndarray
Shape: (4, 384)   (3 sentences, 384 numbers each)

First sentence, first 8 of its 384 numbers:
[-0.047   0.0314  0.0394  0.03   -0.0588  0.0012  0.0354 -0.0348]


Three observations:

1. Every sentence — short or long — becomes the **same length** (384). That is why it is called a *fixed-length* representation: you can stack them in a matrix and do math.
2. The numbers themselves are meaningless to a human. Their *relationships* are what matter.
3. Nothing was stored or searched yet. An embedding model just converts text -> numbers. A **vector database** is what stores those numbers and finds the closest ones — we will build that part by hand next.

## A2 — Similarity is just a number: cosine similarity

To find "the note most like my query", we need to measure how close two vectors are. The standard measure is **cosine similarity**: the cosine of the angle between two vectors.

- Same direction -> cosine **1.0** (most similar)
- Perpendicular -> cosine **0.0** (unrelated)
- Opposite -> cosine **-1.0**

The formula is just the dot product divided by the two lengths:

```
cosine(a, b) = (a . b) / (||a|| x ||b||)
```

It is three numpy operations. Let's write it ourselves.

In [ ]:
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity: dot product over the product of the two vector lengths."""
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

# Compare every pair of the three sentences above
labels = ["pizza-1", "pizza-2", "stocks","current_year"]
print("Pairwise cosine similarity:\n")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        print(f"  {labels[i]:8s} vs {labels[j]:8s} : {cosine(vectors[i], vectors[j]): .3f}")

Pairwise cosine similarity:

  pizza-1  vs pizza-2  :  0.686
  pizza-1  vs stocks   :  0.067
  pizza-1  vs current_year :  0.406
  pizza-2  vs stocks   :  0.005
  pizza-2  vs current_year :  0.477
  stocks   vs current_year :  0.106


**Read the numbers.** The two pizza sentences score high (close to 1) even though they share only the word "pizza". The pizza-vs-stocks pairs score much lower. The model placed "I love eating pizza" and "Pizza is my favourite food" near each other *by meaning*.

A plain keyword search would rank the two pizza sentences as barely related (one shared word). Cosine over embeddings is what makes search feel like it *understands* the query. That single number — and `argsort` over a column of them — is the entire engine of "semantic search".

## A3 — A working semantic search in ~12 lines (no database)

Now the payoff. A vector "database" at its core is: a list of texts, their vectors, and a function that ranks them by cosine to a query. Here is a complete, genuinely useful version with nothing but numpy. Keep this in your back pocket for scripts where pulling in Chroma would be overkill.

In [ ]:
# Our "knowledge base": a handful of personal notes
notes = [
    "RAG means retrieval augmented generation - fetch documents, then generate",
    "Standup is every weekday at 9:30am",
    "Priya prefers design reviews on Tuesday afternoons",
    "Cosine similarity is the dot product of two normalised vectors",
    "Renew the domain name before it expires in August",
    "Vector databases index embeddings for fast nearest-neighbour search",
]

# Embed the whole KB once, up front (shape: 6 x 384)
note_vectors = model.encode(notes)

def semantic_search(query: str, k: int = 3):
    q = model.encode(query)                        # query -> vector (384,)
    scores = [cosine(q, v) for v in note_vectors]  # one cosine per note
    ranked = np.argsort(scores)[::-1][:k]          # indices of top-k, highest first
    return [(notes[i], round(scores[i], 3)) for i in ranked]

for hit, score in semantic_search("how does retrieval augmented generation work?"):
    print(f"{score:.3f}  {hit}")

0.587  RAG means retrieval augmented generation - fetch documents, then generate
0.151  Vector databases index embeddings for fast nearest-neighbour search
0.081  Renew the domain name before it expires in August


The top hit is the RAG note. Now try a query that shares **almost no words** with any stored note — meaning, not vocabulary, is what gets matched:

In [ ]:
# "time" and "meetings" do not appear in the notes — but the meaning does
for hit, score in semantic_search("what time are my meetings?"):

    print(f"{score:.3f}  {hit}")

0.451  Standup is every weekday at 9:30am
0.229  Priya prefers design reviews on Tuesday afternoons
-0.007  Renew the domain name before it expires in August


"what time are my meetings?" surfaces the standup and the design-review notes, even though neither contains the words "time" or "meetings". That is meaning-based retrieval, in twelve lines, with no database.

**One efficiency note for later:** the loop `[cosine(q, v) for v in note_vectors]` is fine for a few hundred notes. For more, you normalise all vectors once and do a single matrix multiply (`note_vectors_normed @ q_normed`) to score everything at once. That is the first thing a real vector DB does for you — and it is why we mention it now.

## A4 — So what does Chroma actually add?

Our hand-rolled search is correct, but it has gaps that show up the moment you leave a toy notebook. Chroma (the store `01_principles` opens in three lines) fills exactly those gaps:

| Concern | Hand-rolled (above) | What Chroma does |
|---|---|---|
| **Persistence** | Vectors vanish when the kernel dies | Writes vectors to disk (`persist_directory`), survives restarts |
| **Speed at scale** | Scores *every* vector on every query (O(n)) | Builds an index for approximate nearest-neighbour — fast at 100k+ |
| **Metadata** | You would hand-roll tag/date filters | `where={"tag": "work"}` filtering alongside the vector search |
| **Add / update / delete** | You mutate Python lists yourself | `add` / `update` / `delete` with stable IDs |
| **Embedding wiring** | You call `model.encode` manually | `embedding_function` runs it for you on add and query |

Nothing here is conceptually new — it is the *same* "embed, then cosine-rank" loop you just wrote, made persistent, fast, and filterable. When you open `01_principles` Section 3 next, you will recognise every piece:

```python
vector_store = Chroma(
    collection_name="personal_notes",
    embedding_function=embeddings,    # <- our model.encode, automated
    persist_directory="./chroma_w12", # <- the persistence we lacked
)
vector_store.similarity_search(query, k=3)   # <- our semantic_search()
```

That is the whole bridge: **Chroma is your 12-line search, productionised.**

# Part B — MCP from the ground up

`02_patterns` jumps to `@mcp.tool()` and a running FastMCP server. Before that pays off, you need to know what problem a "tool protocol" solves and what shape it has. We will build that shape in plain Python first, then meet the real `mcp` SDK doing the identical thing.

## B1 — What does "calling a tool" even mean?

An LLM only does one thing: text in, text out. It cannot read your files or save a note by itself. To let it *act*, your code runs a loop:

1. You tell the model **what tools exist** ("you can call `add_note(text)` or `search_notes(query)`").
2. The model replies with a structured request: *"call `search_notes` with `query='rag'`"*.
3. **Your code** — not the model — actually runs that function and feeds the result back.

Steps 1 and 3 are plumbing *you* own. Before MCP, every app wrote this plumbing from scratch, in its own format. Two ideas do all the work, and they have names:

- **`list_tools()`** — "what can I do?" (step 1)
- **`call_tool(name, arguments)`** — "do this one" (step 3)

Every tool protocol — OpenAI function-calling, LangChain tools, MCP — is a flavour of these two verbs. Let's build them.

In [ ]:
# A complete tool protocol in plain Python. No libraries, no LLM.
# Two verbs: list_tools() and call_tool(name, arguments).

_KB = []   # our note store for this demo

def _add_note(text: str, tags: str = "") -> dict:
    _KB.append({"text": text, "tags": tags})
    return {"ok": True, "stored": len(_KB)}

def _search_notes(query: str, k: int = 3) -> dict:
    hits = [n for n in _KB if query.lower() in n["text"].lower()][:k]
    return {"matches": hits}

# The registry: name -> function + a schema describing its arguments
TOOLS = {
    "add_note": {
        "fn": _add_note,
        "description": "Save a note to the knowledge base",
        "arguments": {"text": "string", "tags": "string (optional)"},
    },
    "search_notes": {
        "fn": _search_notes,
        "description": "Find notes whose text contains the query",
        "arguments": {"query": "string", "k": "integer (optional)"},
    },
}

def list_tools() -> list:
    """Verb 1: advertise what is available (this is what the LLM is shown)."""
    return [{"name": n, "description": t["description"], "arguments": t["arguments"]}
            for n, t in TOOLS.items()]

def call_tool(name: str, arguments: dict) -> dict:
    """Verb 2: dispatch a named call to the real function."""
    if name not in TOOLS:
        return {"error": f"unknown tool: {name}"}
    return TOOLS[name]["fn"](**arguments)

In [ ]:
import json

print("== list_tools() — what the model would be shown ==")
print(json.dumps(list_tools(), indent=2))

print("\n== call_tool(...) — what happens when the model picks one ==")
print(call_tool("add_note", {"text": "RAG = retrieval augmented generation", "tags": "ml"}))
print(call_tool("add_note", {"text": "Dentist on Friday at 4pm"}))
print(call_tool("search_notes", {"query": "rag"}))
print(call_tool("nope", {}))   # unknown tool is handled, not crashed

== list_tools() — what the model would be shown ==
[
  {
    "name": "add_note",
    "description": "Save a note to the knowledge base",
    "arguments": {
      "text": "string",
      "tags": "string (optional)"
    }
  },
  {
    "name": "search_notes",
    "description": "Find notes whose text contains the query",
    "arguments": {
      "query": "string",
      "k": "integer (optional)"
    }
  }
]

== call_tool(...) — what happens when the model picks one ==
{'ok': True, 'stored': 1}
{'ok': True, 'stored': 2}
{'matches': [{'text': 'RAG = retrieval augmented generation', 'tags': 'ml'}]}
{'error': 'unknown tool: nope'}


That is a real, working tool layer in about 25 lines. An agent loop would call `list_tools()` to build its prompt, let the model choose a tool and arguments, then `call_tool(...)` and return the result. **This is genuinely useful on its own** — when everything runs in one Python process and you just need an LLM to reach a few functions, a registry like this is all you need; MCP would be overkill.

So when *do* you need more? When the tools live somewhere else — a different process, a different machine, or someone else's app (Claude Desktop, Cursor). That is the moment a *protocol* — an agreed message format on the wire — earns its keep.

## B2 — Putting the two verbs "on the wire": JSON-RPC

When the caller and the tool are in different processes, they cannot share Python objects — they exchange **text messages**. MCP uses a tiny, decades-old convention called **JSON-RPC 2.0** for those messages. "JSON-RPC" sounds heavy; it is just a dict with a few predictable keys.

A request to call a tool looks like this:

```json
{ "jsonrpc": "2.0", "id": 1, "method": "tools/call",
  "params": { "name": "search_notes", "arguments": { "query": "rag" } } }
```

and the reply pairs back by `id`:

```json
{ "jsonrpc": "2.0", "id": 1, "result": { "matches": [ ] } }
```

`method` is the verb (`tools/list` or `tools/call`), `params` carries the arguments, and `id` lets the caller match each reply to its request. That is the *entire* idea. Watch our same two functions answer a JSON-RPC message:

In [ ]:
def handle_jsonrpc(message: dict) -> dict:
    """Route a JSON-RPC message to our list_tools / call_tool. This is, in miniature,
    what an MCP server does for every request that arrives on its transport."""
    mid = message.get("id")
    method = message.get("method")
    params = message.get("params", {})

    if method == "tools/list":
        result = list_tools()
    elif method == "tools/call":
        result = call_tool(params["name"], params.get("arguments", {}))
    else:
        return {"jsonrpc": "2.0", "id": mid,
                "error": {"message": f"unknown method {method}"}}

    return {"jsonrpc": "2.0", "id": mid, "result": result}

# A client would send this text; a server would send the reply text back
request = {"jsonrpc": "2.0", "id": 1, "method": "tools/call",
           "params": {"name": "search_notes", "arguments": {"query": "rag"}}}

print("REQUEST :", json.dumps(request))
print("RESPONSE:", json.dumps(handle_jsonrpc(request)))

REQUEST : {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search_notes", "arguments": {"query": "rag"}}}
RESPONSE: {"jsonrpc": "2.0", "id": 1, "result": {"matches": [{"text": "RAG = retrieval augmented generation", "tags": "ml"}]}}


You have now built, by hand, the core of an MCP server: a `tools/list` + `tools/call` dispatcher speaking JSON-RPC. The only things missing are (a) a **transport** to carry those messages between processes (MCP uses stdio or HTTP), and (b) a standard so *any* client knows the method names without you documenting them. That standard, plus the transport, is what the `mcp` library provides — so you never write `handle_jsonrpc` again.

## B3 — The same thing with the real library: FastMCP

`FastMCP` turns a decorated Python function into a registered tool and answers the `tools/list` / `tools/call` traffic for you. Below we build a *minimal* server in the notebook and call it through the real SDK's async API, so you can see the two verbs you just wrote, now coming from the library itself.

In [ ]:
# The real library. FastMCP is the high-level server from the `mcp` package.
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("foundations-demo")

@mcp.tool()
def add_note(text: str, tags: str = "") -> dict:
    """Save a note to the knowledge base."""
    _KB.append({"text": text, "tags": tags})
    return {"ok": True, "stored": len(_KB)}

@mcp.tool()
def search_notes(query: str, k: int = 3) -> dict:
    """Find notes whose text contains the query."""
    hits = [n for n in _KB if query.lower() in n["text"].lower()][:k]
    return {"matches": hits}

print("FastMCP server created. The @mcp.tool() decorator registered 2 tools.")
print("Notice: same two functions as our hand-rolled registry — just decorated.")

FastMCP server created. The @mcp.tool() decorator registered 2 tools.
Notice: same two functions as our hand-rolled registry — just decorated.


### Calling the server through the real SDK

FastMCP exposes the same two verbs as **async** methods: `await mcp.list_tools()` and `await mcp.call_tool(name, arguments)`. The `mcp` library is async because real transports (stdio, HTTP) are async.

Jupyter and Colab let you use `await` at the top level of a cell, so the next cell awaits directly. (In a plain `.py` script you would wrap these in `async def main(): ...` and run `asyncio.run(main())`.)

The return shape of `call_tool` has varied a little across `mcp` versions, so we use a tiny `extract(...)` helper to pull the data out regardless of version — a good habit when calling fast-moving SDKs.

In [ ]:
def extract(call_result):
    """Normalise FastMCP.call_tool() return across mcp versions into something printable."""
    content = call_result
    if isinstance(call_result, tuple):      # some versions: (content, structured)
        content = call_result[0]
    if hasattr(content, "content"):         # some versions: a CallToolResult object
        content = content.content
    out = []
    for part in content:
        out.append(getattr(part, "text", None) or getattr(part, "data", None) or str(part))
    return out

# Verb 1 — list_tools(): the library advertises the decorated tools
tools = await mcp.list_tools()
print("== mcp.list_tools() ==")
for t in tools:
    print(f"  {t.name}: {t.description}")

# Verb 2 — call_tool(): the library dispatches to our function
print("\n== mcp.call_tool('add_note', ...) ==")
print(extract(await mcp.call_tool("add_note", {"text": "MCP standardises tools/list and tools/call"})))

print("\n== mcp.call_tool('search_notes', ...) ==")
print(extract(await mcp.call_tool("search_notes", {"query": "mcp"})))

== mcp.list_tools() ==
  add_note: Save a note to the knowledge base.
  search_notes: Find notes whose text contains the query.

== mcp.call_tool('add_note', ...) ==
['{\n  "ok": true,\n  "stored": 3\n}']

== mcp.call_tool('search_notes', ...) ==
['{\n  "matches": [\n    {\n      "text": "MCP standardises tools/list and tools/call",\n      "tags": ""\n    }\n  ]\n}']


> **If your `mcp` version errors on the cell above:** the public way to exercise a server is the inspector — `mcp dev mcp_server.py` (shown in `02_patterns`) — or the "impl + tool wrapper" test harness in `02_patterns` Section 7, which calls the plain Python function directly. The two verbs are the same either way; only the call surface differs.

## B4 — So what does MCP add over our registry?

Same two verbs, so what is the `mcp` library buying you? Exactly the things that are painful to hand-roll once tools leave your process:

| Concern | Hand-rolled registry | What MCP / FastMCP adds |
|---|---|---|
| **Where tools run** | Same Python process only | Separate process or machine, over stdio or HTTP |
| **Message format** | You invented `handle_jsonrpc` | Standard JSON-RPC method names every client knows |
| **Schemas** | You wrote `"arguments": {...}` by hand | Generated from type hints, validated automatically |
| **More than tools** | Only functions | Also **resources** (read-only context) and **prompts** (user-invokable templates) |
| **Interoperability** | Only your app can call it | Claude Desktop, Cursor, the MCP inspector — any MCP client |

The decorator did not introduce a new idea. It wrapped the `list_tools` / `call_tool` handshake you built in B1, put it on a real transport, and made it speak a format other apps already understand. That is why `02_patterns` ships `mcp_server.py` as a *separate file*: the whole point is that something *outside* your notebook connects to it.

## Recap — the machinery behind the next two notebooks

**Vector DBs (Part A).** An embedding turns text into a fixed-length vector; cosine similarity scores how aligned two vectors are; semantic search is "embed the query, rank stored vectors by cosine, take the top-k". You wrote that in 12 lines. **Chroma** in `01_principles` is that same loop made persistent, fast, and filterable.

**MCP (Part B).** Letting an LLM act needs two verbs: `list_tools` (what can I do?) and `call_tool` (do this). Put them on the wire as JSON-RPC and you have a protocol. You wrote that by hand. **FastMCP** in `02_patterns` is that same handshake over a real transport, with schemas, resources, prompts, and any-client interoperability.

Neither Chroma nor FastMCP is magic — each is a well-engineered wrapper around a small idea you can now implement yourself. Carry that lens into `01_principles` and `02_patterns`: every "convenience" you meet there maps back to something in this notebook.

**Next:** open `01_principles.ipynb` (memory types + Chroma), then `02_patterns.ipynb` (the real FastMCP server).